# Ejercicio 1 - Introducción a RAG (Retrieval Augmented Generation)

¡Bienvenido/a! Este cuaderno te guiará paso a paso para entender **RAG** desde cero, usando solo **CPU**.

## ¿Qué aprenderás?

1. Crear una mini base de conocimiento (corpus) desde cero.
2. Convertir textos a vectores numéricos (**embeddings**) que capturen su significado.
3. Almacenar y buscar esos vectores con **FAISS** (base vectorial).
4. Responder preguntas de dos formas:
   - **Sin RAG** (el modelo responde solo con lo que aprendió durante su entrenamiento).
   - **Con RAG** (el modelo utiliza información recuperada de tu base de conocimiento).
5. Comparar resultados y evaluarlos de forma sencilla.

## ¿Qué es Hugging Face y por qué lo usamos?

Hugging Face es una plataforma y conjunto de librerías que ofrece modelos de lenguaje preentrenados. Aquí usamos dos de sus piezas:

- `transformers`: para cargar un modelo generativo (el que redacta respuestas).
- `sentence-transformers`: para cargar un modelo que convierte texto en vectores (embeddings).

Esto nos permite **reutilizar** modelos potentes sin tener que entrenarlos desde cero.

## ¿Qué es una base vectorial y por qué FAISS?

Una base vectorial busca textos por **similitud semántica**, no por coincidencia exacta de palabras.

1. Cada documento se convierte en un vector (un array de números).
2. La pregunta del usuario también se convierte en vector.
3. Se recuperan los documentos cuyos vectores están más cerca (son más parecidos) al vector de la pregunta.

**FAISS** (Facebook AI Similarity Search) es una librería rápida y sencilla que hace esta búsqueda.

> **Objetivo didáctico**: entender por qué RAG mejora las respuestas cuando el conocimiento no está dentro del modelo generativo.


## Librerías utilizadas (¿qué aporta cada una?)

- `torch`: motor numérico que corre los modelos (deep learning).
- `transformers`: nos da el modelo generativo (`pipeline`).
- `sentence-transformers`: modelo de embeddings para convertir texto a vectores.
- `faiss-cpu`: índice vectorial para búsqueda de vecinos más cercanos (solo CPU).
- `pandas`: para crear tablas de resultados y evaluaciones.
- `IPython.display`: para mostrar tablas bonitas en el notebook.

> **Nota**: Elegimos versiones ligeras para que funcionen en CPU y en tiempos razonables para clase.

In [ ]:
# Instalación de dependencias (solo si tu entorno no las tiene)
# Ejecuta esta celda UNA VEZ al inicio del laboratorio.
# Si ya tienes las librerías instaladas, puedes comentar la línea con %pip.

# %pip install -q transformers torch sentence-transformers faiss-cpu pandas

In [ ]:
# Importamos las librerías necesarias
import pandas as pd
import torch
import faiss
import re

from transformers import pipeline
from sentence_transformers import SentenceTransformer
from IPython.display import display

print('Versión de Torch:', torch.__version__)
print('¿Usando CPU?', not torch.cuda.is_available())
print('Todo listo para empezar con RAG.')

## Paso 1 - Construir la base de conocimiento (corpus)

Aquí simulamos una pequeña base de documentos. En la vida real podrían venir de PDFs, páginas web, bases de datos internas, etc.

**¡Importante!** La calidad del RAG depende muchísimo de esta base: si falta información, no se podrá recuperar; si hay información incorrecta, la respuesta será mala aunque la recuperación sea buena.

Nuestro corpus trata sobre políticas de una empresa ficticia llamada **UMA Tech**.

In [ ]:
corpus = [
    'UMA Tech permite trabajo remoto hasta 3 dias por semana para perfiles de desarrollo y analitica.',
    'El horario flexible permite entrada entre 7:00 y 10:00, completando 8 horas laborales.',
    'La empresa cubre 70% del costo de certificaciones tecnicas aprobadas por el lider de area.',
    'Los colaboradores tienen 20 dias habiles de vacaciones al ano mas 2 dias personales.',
    'La licencia por paternidad es de 15 dias calendario y la licencia por maternidad sigue la ley local.',
    'El presupuesto anual de formacion por empleado es de 600 USD.',
    'Para solicitar equipos nuevos se debe abrir ticket en Mesa TI y obtener aprobacion del manager.',
    'No hay politica oficial de semana laboral de 4 dias en UMA Tech.'
]

print('Base de conocimiento (corpus):')
for i, texto in enumerate(corpus, start=1):
    print(f'{i}. {texto}')

## Paso 2 - Convertir documentos a vectores (embeddings)

Un embedding es un vector numérico que representa el **significado** de un texto. Textos con significado similar tendrán vectores cercanos en el espacio.

Usamos `sentence-transformers/all-MiniLM-L6-v2`, un modelo pequeño y rápido que funciona bien en CPU y es perfecto para aprender.

In [ ]:
nombre_modelo_emb = 'sentence-transformers/all-MiniLM-L6-v2'
print(f'Cargando modelo de embeddings: {nombre_modelo_emb} ...')
embedder = SentenceTransformer(nombre_modelo_emb)

embeddings = embedder.encode(corpus, convert_to_numpy=True).astype('float32')
print('Vectores generados con forma:', embeddings.shape)
print(f'Tenemos {embeddings.shape[0]} documentos y cada vector tiene {embeddings.shape[1]} dimensiones.')

## Paso 3 - Crear la base vectorial con FAISS

FAISS nos permite guardar los vectores y buscar los más cercanos a una consulta de forma eficiente.

Usamos `IndexFlatL2`:
- **L2** = distancia euclidiana (la raíz de la suma de cuadrados de las diferencias).
- Es un índice sencillo y exacto, ideal para aprender.

In [ ]:
dimension = embeddings.shape[1]
indice_faiss = faiss.IndexFlatL2(dimension)
indice_faiss.add(embeddings)

print('Número de vectores indexados en FAISS:', indice_faiss.ntotal)

## Paso 4 - Cargar el modelo generativo (el que escribe respuestas)

Usamos un modelo pequeño llamado `sshleifer/tiny-gpt2`. Es muy ligero y corre en CPU. No esperes respuestas increíbles, pero sirve perfecto para entender el flujo RAG.

Más adelante podrás probar modelos más grandes si quieres mejor calidad.

In [ ]:
nombre_modelo_gen = 'sshleifer/tiny-gpt2'
print(f'Cargando modelo generativo: {nombre_modelo_gen} ...')
generador = pipeline(
    'text-generation',
    model=nombre_modelo_gen,
    tokenizer=nombre_modelo_gen,
    device=-1
)
print('Modelo generativo listo.')
print('Tarea:', generador.task)

## Paso 5 - Funciones para el flujo RAG

Aquí definimos las piezas clave:

1. `recuperar_contexto`: busca documentos relevantes en FAISS dada una pregunta.
2. `generar_respuesta`: función interna que usa el modelo generativo.
3. `extraer_respuesta_de_docs`: versión extractiva (sencilla) para mostrar claramente el impacto del contexto.
4. `responder_sin_rag`: línea base sin contexto externo.
5. `responder_con_rag`: flujo completo (recuperar + generar).

> **Filosofía RAG**: primero **recuperamos** información relevante, luego **generamos** la respuesta basada en esa información.

In [ ]:
def recuperar_contexto(pregunta: str, top_k: int = 2, verbose: bool = False):
    emb_pregunta = embedder.encode([pregunta], convert_to_numpy=True).astype('float32')
    distancias, indices = indice_faiss.search(emb_pregunta, top_k)
    docs_recuperados = [corpus[i] for i in indices[0]]
    if verbose:
        print('\n[RECUPERACIÓN]')
        print(f'Pregunta: "{pregunta}"')
        print(f'Top_k = {top_k}')
        for rank, (idx, dist, doc) in enumerate(zip(indices[0], distancias[0], docs_recuperados), start=1):
            print(f'  Documento {rank} (índice {idx}) - distancia L2 = {dist:.4f}')
            print(f'    Texto: "{doc}"')
    return docs_recuperados, distancias[0], indices[0]

def _generar(prompt: str, max_tokens_nuevos: int = 60):
    salida = generador(
        prompt,
        max_new_tokens=max_tokens_nuevos,
        do_sample=False,
        return_full_text=False
    )[0]['generated_text']
    return salida.strip()

def extraer_respuesta_de_docs(pregunta: str, docs: list):
    contexto = ' '.join(docs)
    q = pregunta.lower()
    if 'trabajo remoto' in q:
        m = re.search(r'hasta\s+(\d+)\s+dias', contexto.lower())
        if m:
            return f'UMA Tech permite trabajo remoto hasta {m.group(1)} dias por semana.'
    if 'presupuesto anual' in q or 'formacion' in q:
        m = re.search(r'(\d+\s*usd)', contexto.lower())
        if m:
            return f'El presupuesto anual de formacion por empleado es de {m.group(1).upper()}.'
    if '4 dias' in q or 'semana laboral' in q:
        if 'no hay politica oficial' in contexto.lower():
            return 'No hay politica oficial de semana laboral de 4 dias en UMA Tech.'
    return docs[0] if docs else 'No tengo contexto suficiente para responder.'

def responder_sin_rag(pregunta: str, max_tokens_nuevos: int = 60, verbose: bool = False):
    prompt = (
        'Responde de forma breve y honesta. '
        'Si no sabes la respuesta exacta, dilo.\n\n'
        f'Pregunta: {pregunta}\nRespuesta:'
    )
    if verbose:
        print('\n[SIN RAG] Generando respuesta sin contexto externo...')
    return _generar(prompt, max_tokens_nuevos=max_tokens_nuevos)

def responder_con_rag(pregunta: str, top_k: int = 2, max_tokens_nuevos: int = 80, verbose: bool = False):
    docs, distancias, indices = recuperar_contexto(pregunta, top_k=top_k, verbose=verbose)
    respuesta_extraida = extraer_respuesta_de_docs(pregunta, docs)
    contexto = '\n'.join([f'- {d}' for d in docs])
    prompt = (
        'Usa SOLO el contexto para responder de forma breve en español. '
        'Si el contexto no alcanza, dilo explícitamente.\n\n'
        f'Contexto:\n{contexto}\n\n'
        f'Pregunta: {pregunta}\nRespuesta:'
    )
    respuesta_generada = _generar(prompt, max_tokens_nuevos=max_tokens_nuevos)
    if verbose:
        print('\n[CON RAG] Respuesta extraída del contexto:', respuesta_extraida)
        print('[CON RAG] Respuesta generada por el LLM    :', respuesta_generada)
    return respuesta_extraida, docs, distancias, indices, respuesta_generada

## Paso 6 - Experimento comparativo: Sin RAG vs Con RAG

Vamos a probar **tres preguntas** que sí están cubiertas en nuestro corpus. Así veremos la diferencia entre:

- **Sin RAG**: el modelo responde solo con su conocimiento interno (muy limitado porque es un modelo pequeño).
- **Con RAG (extractivo)**: respuesta basada directamente en los documentos recuperados.
- **Con RAG (generativo)**: el LLM redacta una respuesta usando el contexto.

Observa cómo el modelo sin RAG a menudo no sabe la respuesta o inventa, mientras que con RAG consigue responder correctamente porque le damos la información.

In [ ]:
preguntas = [
    'Cuantos dias de trabajo remoto permite UMA Tech?',
    'Cual es el presupuesto anual de formacion por empleado?',
    'Existe politica oficial de semana laboral de 4 dias?'
]

resultados = []

print('=' * 90)
print('EXPERIMENTO: Sin RAG vs Con RAG')
print('=' * 90)

for i, pregunta in enumerate(preguntas, start=1):
    print(f'\n--- Caso {i}/{len(preguntas)} ---')
    print(f'Pregunta: "{pregunta}"')
    sin_rag = responder_sin_rag(pregunta, verbose=True)
    con_rag_ext, docs, distancias, indices, con_rag_llm = responder_con_rag(pregunta, top_k=2, verbose=True)
    resultados.append({
        'pregunta': pregunta,
        'respuesta_sin_rag': sin_rag,
        'respuesta_con_rag_extractiva': con_rag_ext,
        'respuesta_con_rag_generativa': con_rag_llm,
        'docs_recuperados': ' || '.join(docs)
    })
    print('\n[RESUMEN DE ESTE CASO]')
    print(f'  SIN RAG: {sin_rag}')
    print(f'  CON RAG (extractivo): {con_rag_ext}')
    print(f'  CON RAG (LLM): {con_rag_llm}')
    print('  Documentos recuperados:')
    for d in docs:
        print(f'    - {d}')

print('\n' + '=' * 90)
print('EXPERIMENTO FINALIZADO')
print('=' * 90)

## Paso 7 - Evaluación simple de los resultados

Para medir si las respuestas son correctas, usaremos una métrica muy sencilla: ¿contiene la respuesta el fragmento esperado?

Esto nos permite calcular una **accuracy** (precisión) para el modo sin RAG y con RAG y ver la mejora.

> **Nota**: En un sistema real usaríamos métricas más avanzadas, pero para aprender esta aproximación es suficiente.

In [ ]:
esperados = {
    'Cuantos dias de trabajo remoto permite UMA Tech?': '3 dias por semana',
    'Cual es el presupuesto anual de formacion por empleado?': '600 USD',
    'Existe politica oficial de semana laboral de 4 dias?': 'No hay politica oficial'
}

def puntuar_contiene(respuesta: str, fragmento_esperado: str) -> int:
    return int(fragmento_esperado.lower() in respuesta.lower())

filas_eval = []
print('Evaluando resultados...\n')
for r in resultados:
    pregunta = r['pregunta']
    esperado = esperados[pregunta]
    acierto_sin = puntuar_contiene(r['respuesta_sin_rag'], esperado)
    acierto_con = puntuar_contiene(r['respuesta_con_rag_extractiva'], esperado)
    filas_eval.append({
        'Pregunta': pregunta,
        'Esperado (contiene)': esperado,
        '¿Acierto sin RAG?': 'Sí' if acierto_sin else 'No',
        '¿Acierto con RAG?': 'Sí' if acierto_con else 'No'
    })

df_eval = pd.DataFrame(filas_eval)
display(df_eval)

acc_sin = sum(1 for r in resultados if puntuar_contiene(r['respuesta_sin_rag'], esperados[r['pregunta']])) / len(resultados)
acc_con = sum(1 for r in resultados if puntuar_contiene(r['respuesta_con_rag_extractiva'], esperados[r['pregunta']])) / len(resultados)

print('\n--- Precisión global ---')
print(f'Sin RAG: {acc_sin*100:.0f}%')
print(f'Con RAG (extractivo): {acc_con*100:.0f}%')
print(f'Mejora: {(acc_con - acc_sin)*100:.0f}%')

if acc_con > acc_sin:
    print('\n✅ El sistema con RAG ha mejorado notablemente. ¡Bien!')
else:
    print('\n⚠️ No hubo mejora, revisa el retrieval o el corpus.')

## Paso 8 - Métricas de calidad: RAG Triad (para alumnos avanzados)

En la evaluación de sistemas RAG se suele usar un conjunto de tres métricas llamadas **RAG Triad**:

1. **Context Relevance (Relevancia del contexto)**: ¿Los documentos recuperados son útiles para responder la pregunta? Lo medimos como la proporción de palabras clave de la pregunta que aparecen en los documentos.
2. **Faithfulness (Fidelidad)**: ¿La respuesta está respaldada por el contexto (vs alucinaciones)? Lo medimos como la proporción de palabras clave de la respuesta que están en los documentos.
3. **Answer Relevance (Relevancia de la respuesta)**: ¿La respuesta realmente responde a lo que se preguntó? Lo medimos por la presencia del fragmento esperado.

Estas métricas son **simplificadas** para el aula (usamos reglas heurísticas), pero te dan una idea de cómo evaluar un pipeline RAG de forma más fina.

In [ ]:
stopwords_es = {
    'el', 'la', 'los', 'las', 'de', 'del', 'y', 'o', 'a', 'en', 'por', 'para',
    'con', 'sin', 'es', 'un', 'una', 'que', 'cual', 'cuantos', 'cuantas',
    'permite', 'existe', 'politica', 'oficial', 'semana', 'laboral'
}

def normalizar_texto(texto: str) -> str:
    return re.sub(r'\s+', ' ', texto.lower().strip())

def conjunto_palabras_clave(texto: str):
    tokens = re.findall(r'[a-z0-9]+', normalizar_texto(texto))
    return {t for t in tokens if len(t) > 2 and t not in stopwords_es}

def metric_context_relevance(pregunta: str, docs_texto: str) -> float:
    kw_pregunta = conjunto_palabras_clave(pregunta)
    kw_docs = conjunto_palabras_clave(docs_texto)
    if not kw_pregunta:
        return 0.0
    return len(kw_pregunta & kw_docs) / len(kw_pregunta)

def metric_faithfulness(respuesta: str, docs_texto: str) -> float:
    kw_respuesta = conjunto_palabras_clave(respuesta)
    kw_docs = conjunto_palabras_clave(docs_texto)
    if not kw_respuesta:
        return 0.0
    return len(kw_respuesta & kw_docs) / len(kw_respuesta)

def metric_answer_relevance(pregunta: str, respuesta: str, fragmento_esperado: str) -> float:
    return float(fragmento_esperado.lower() in respuesta.lower())

print('\nEvaluación RAG Triad (usando respuestas extractivas)')
filas_triad = []
for r in resultados:
    pregunta = r['pregunta']
    respuesta = r['respuesta_con_rag_extractiva']
    docs_texto = r['docs_recuperados'].replace(' || ', ' ')
    esperado = esperados[pregunta]
    rel_contexto = metric_context_relevance(pregunta, docs_texto)
    fidelidad = metric_faithfulness(respuesta, docs_texto)
    rel_respuesta = metric_answer_relevance(pregunta, respuesta, esperado)
    triad_score = (rel_contexto + fidelidad + rel_respuesta) / 3
    filas_triad.append({
        'Pregunta': pregunta,
        'Context Relevance': round(rel_contexto, 2),
        'Faithfulness': round(fidelidad, 2),
        'Answer Relevance': round(rel_respuesta, 2),
        'Triad Score': round(triad_score, 2)
    })

df_triad = pd.DataFrame(filas_triad)
display(df_triad)

print('\n--- Interpretación de la RAG Triad ---')
print(' - Context Relevance alto (>0.7): el retrieval trajo documentos relevantes.')
print(' - Faithfulness alto (>0.7): la respuesta está bien apoyada en el contexto (poca alucinación).')
print(' - Answer Relevance alto (>0.7): la respuesta responde directamente a la pregunta.')
print(' - Triad Score: promedio de las tres. Si es bajo, revisa retrieval o el prompt.')

## Análisis y preguntas para reflexionar

- ¿Qué parte del pipeline RAG crees que es más crítica: los embeddings, la base vectorial, el número de documentos recuperados (top_k) o el prompt?
- ¿Qué pasaría si el corpus fuera mucho más grande (e.g., 10,000 documentos)? ¿Seguiría funcionando bien FAISS?
- ¿Qué riesgos de alucinación persisten incluso usando RAG?

## Retos para seguir aprendiendo

1. **Cambia `top_k`** a 1, 3 o 5 y observa cómo cambian las respuestas.
2. **Agrega 5 documentos nuevos** al corpus (por ejemplo, sobre beneficios, horarios, etc.) y prueba nuevas preguntas.
3. **Prueba otro modelo de embeddings** como `'paraphrase-MiniLM-L3-v2'` y compara la calidad del retrieval.
4. **Modifica el prompt** en `responder_con_rag` para que exija respuestas de exactamente una frase.
5. **Crea una pregunta cuya respuesta NO esté en el corpus** y analiza cómo se comporta el sistema con RAG.
6. **Sustituye el modelo generativo** por otro pequeño como `'distilgpt2'` (requiere más RAM, pero funciona en CPU).
7. **Introduce dos documentos contradictorios** en el corpus y observa cómo afecta a la respuesta final.

---
¡Has completado el ejercicio introductorio a RAG! Ahora entiendes cómo combinar recuperación de información con generación de lenguaje.